In [1]:
!pip install pyswarms scikeras

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 4.7 MB/s eta 0:00:00


In [2]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
import pyswarms as ps

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Mengatur seed agar hasil bisa direproduksi (reproducible)
np.random.seed(42)
tf.random.set_seed(42)

# Buat folder untuk menyimpan parameter terbaik
out_dir = '/kaggle/working/outputs'
os.makedirs(out_dir, exist_ok=True)
print("Setup Selesai!")

2026-06-09 20:10:56.401749: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781035856.640568      21 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781035856.703948      21 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781035857.306129      21 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781035857.306173      21 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781035857.306176      21 computation_placer.cc:177] computation placer alr

Setup Selesai!


In [ ]:
# Karena kita menggunakan Cross Validation, kita gabungkan Train dan Val untuk dievaluasi oleh Sliding Window
train_path = '/kaggle/input/notebooks/fauzanrasendriya/01-data-preprocessing-eda/opstat_1/train.csv'
val_path = '/kaggle/input/notebooks/fauzanrasendriya/01-data-preprocessing-eda/opstat_1/val.csv'

df_train = pd.read_csv(train_path, parse_dates=['Tanggal'], index_col='Tanggal')
df_val = pd.read_csv(val_path, parse_dates=['Tanggal'], index_col='Tanggal')

# Gabungkan data untuk Cross Validation
df_cv = pd.concat([df_train, df_val])

# Normalisasi Data
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(df_cv[['Y']].values)

print(f"Total data untuk Sliding Window CV: {len(scaled_data)} hari.")

Total data untuk Sliding Window CV: 3615 hari.


In [4]:
# Fungsi untuk membuat sekuens X dan y berdasarkan lag (seq_len)
def create_sequences(dataset, seq_len):
    X, y = [], []
    for i in range(len(dataset) - seq_len):
        X.append(dataset[i:(i + seq_len), 0])
        y.append(dataset[i + seq_len, 0])
    return np.array(X), np.array(y)

# Fungsi untuk merakit arsitektur LSTM secara dinamis
def build_lstm(seq_len, hidden_size, num_layers, lr, dropout):
    model = Sequential()
    # Layer 1
    model.add(LSTM(hidden_size, return_sequences=(num_layers > 1), input_shape=(seq_len, 1)))
    model.add(Dropout(dropout))
    
    # Tambahan Layer jika num_layers > 1
    for i in range(1, num_layers):
        model.add(LSTM(hidden_size, return_sequences=(i < num_layers - 1)))
        model.add(Dropout(dropout))
        
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=lr), loss='mse')
    return model

In [5]:
# Setup Sliding Window (Membatasi max_train_size agar data terlama dibuang)
batas_hari_latih = 500 
tscv = TimeSeriesSplit(n_splits=3, max_train_size=batas_hari_latih)

def pso_objective_function(particles):
    """
    Menghitung fitness (RMSE) untuk setiap partikel dalam swarm.
    Urutan dimensi: [seq_len, hidden_size, num_layers, lr, dropout, batch_size, epochs]
    """
    n_particles = particles.shape[0]
    errors = np.zeros(n_particles)
    
    for i in range(n_particles):
        seq_len     = int(particles[i, 0])
        hidden_size = int(particles[i, 1])
        num_layers  = int(particles[i, 2])
        lr          = particles[i, 3]
        dropout     = particles[i, 4]
        batch_size  = int(particles[i, 5])
        epochs      = int(particles[i, 6])
        
        # 2. Buat Sekuens Data dengan seq_len milik partikel ini
        X, y = create_sequences(scaled_data, seq_len)
        X = X.reshape(X.shape[0], X.shape[1], 1)
        
        fold_rmse = []
        
        # 3. Sliding Window Cross Validation
        for train_index, test_index in tscv.split(X):
            X_train, X_test = X[train_index], X[test_index]
            y_train, y_test = y[train_index], y[test_index]
            
            # Bangun dan Latih Model
            model = build_lstm(seq_len, hidden_size, num_layers, lr, dropout)
            model.fit(X_train, y_train, batch_size=batch_size, epochs=epochs, verbose=0)
            
            # Prediksi dan Hitung Error
            predictions = model.predict(X_test, verbose=0)
            rmse = np.sqrt(mean_squared_error(y_test, predictions))
            fold_rmse.append(rmse)
        
        # Fitness partikel adalah Rata-rata RMSE dari semua Folds
        avg_rmse = np.mean(fold_rmse)
        errors[i] = avg_rmse
        
        print(f"Partikel {i+1} | RMSE_CV: {avg_rmse:.5f} | seq: {seq_len}, hid: {hidden_size}, lr: {lr:.4f}, ep: {epochs}")
        
    return errors

In [6]:
# Batas Minimum (Lower Bounds)
# [seq_len, hidden_size, num_layers, lr, dropout, batch_size, epochs]
min_bounds = np.array([10,  50,  1, 0.0001, 0.0, 16,  30])

# Batas Maksimum (Upper Bounds)
max_bounds = np.array([50, 300,  3, 0.0100, 0.5, 128, 100])

bounds = (min_bounds, max_bounds)

# Konfigurasi Algoritma PSO
# c1 (kognitif), c2 (sosial), w (inersia)
options = {'c1': 1.5, 'c2': 1.5, 'w': 0.7}

# Budget Komputasi (Jumlah evaluasi model = n_particles * iters)
n_particles = 20
iters = 30 

print("================================================================")
print("MEMULAI CPSO DENGAN SLIDING WINDOW CV")
print(f"Total Partikel: {n_particles} | Iterasi: {iters} | Folds: {tscv.n_splits}")
print("================================================================")

# Panggil Optimizer
optimizer = ps.single.GlobalBestPSO(
    n_particles=n_particles, 
    dimensions=7, 
    options=options, 
    bounds=bounds
)

# Jalankan Proses Optimasi
best_cost, best_pos = optimizer.optimize(pso_objective_function, iters=iters)

2026-06-09 20:11:10,011 - pyswarms.single.global_best - INFO - Optimize for 30 iters with {'c1': 1.5, 'c2': 1.5, 'w': 0.7}


MEMULAI CPSO DENGAN SLIDING WINDOW CV
Total Partikel: 20 | Iterasi: 30 | Folds: 3


pyswarms.single.global_best:   0%|          |0/30I0000 00:00:1781035871.849121      21 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1781035871.855566      21 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
I0000 00:00:1781035875.100780      73 cuda_dnn.cc:529] Loaded cuDNN version 91002
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a 

Partikel 1 | RMSE_CV: 0.02972 | seq: 45, hid: 134, lr: 0.0010, ep: 62


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


...

Partikel 18 | RMSE_CV: 0.06593 | seq: 31, hid: 156, lr: 0.0073, ep: 45


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Partikel 19 | RMSE_CV: 0.05514 | seq: 35, hid: 190, lr: 0.0097, ep: 33


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
pyswarms.single.global_best: 100%|██████████|30/30, best_cost=0.0133
2026-06-10 01:10:50,061 - pyswarms.single.global_best - INFO - Optimizati

Partikel 20 | RMSE_CV: 0.03820 | seq: 30, hid: 223, lr: 0.0040, ep: 82


In [7]:
# Ekstrak Parameter Terbaik
best_hp = {
    "seq_len": int(best_pos[0]),
    "hidden_size": int(best_pos[1]),
    "num_layers": int(best_pos[2]),
    "lr": float(best_pos[3]),
    "dropout": float(best_pos[4]),
    "batch_size": int(best_pos[5]),
    "epochs": int(best_pos[6])
}

print("============================================================")
print("RINGKASAN CPSO (SLIDING WINDOW CV)")
print("============================================================")
print(f"Best RMSE_CV (Normalized): {best_cost:.6f}")
print("Hyperparameter Optimal:")
for k, v in best_hp.items():
    print(f"  - {k:<15}: {v}")

# Simpan ke JSON
results_path = os.path.join(out_dir, "cpso_sliding_results.json")
with open(results_path, "w") as f:
    json.dump(best_hp, f, indent=4)

print(f"\nHasil CPSO berhasil disimpan ke: {results_path}")
print("Anda dapat menggunakan parameter ini di Notebook 4 (Final Training).")

RINGKASAN CPSO (SLIDING WINDOW CV)
Best RMSE_CV (Normalized): 0.013256
Hyperparameter Optimal:
  - seq_len        : 29
  - hidden_size    : 151
  - num_layers     : 1
  - lr             : 0.008513645498481031
  - dropout        : 0.09459040576607597
  - batch_size     : 19
  - epochs         : 87

Hasil CPSO berhasil disimpan ke: /kaggle/working/outputs/cpso_sliding_results.json
Anda dapat menggunakan parameter ini di Notebook 4 (Final Training).
